In [10]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [11]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

from src.data_preprocessing import load_data
from src.model import GCN, APPNPModel, GPRGNN
from src.train import train, test

PHASE 1 - ARCHITECTURE COMPARISON 

In [12]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Fixed baseline hyperparameters
hidden_dim = 64
K = 10
alpha = 0.2
dropout = 0.5
weight_decay = 5e-4
lr = 0.01
epochs = 300
split_type = "random_80_10_10"

seeds = [0, 42, 7, 123, 999]

print("Using device:", device)

Using device: mps


In [13]:
dataset, data = load_data(device, split_type=split_type)

print("Number of nodes:", data.num_nodes)
print("Number of features:", dataset.num_node_features)
print("Number of classes:", dataset.num_classes)

Number of nodes: 3327
Number of features: 3703
Number of classes: 6


In [14]:
def run_model(model_name):
    results = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        if model_name == "gcn":
            model = GCN(
                dataset.num_node_features,
                hidden_dim,
                dataset.num_classes,
                dropout=dropout
            ).to(device)

        elif model_name == "appnp":
            model = APPNPModel(
                dataset.num_node_features,
                hidden_dim,
                dataset.num_classes,
                K=K,
                alpha=alpha,
                dropout=dropout
            ).to(device)

        elif model_name == "gpr":
            model = GPRGNN(
                dataset.num_node_features,
                hidden_dim,
                dataset.num_classes,
                K=K,
                alpha=alpha,
                dropout=dropout
            ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        for epoch in range(epochs):
            train(model, data, optimizer)

        train_acc, val_acc, test_acc = test(model, data)

        results.append(test_acc)

    mean_acc = np.mean(results)
    std_acc = np.std(results)

    return mean_acc, std_acc

In [15]:
mean_gcn, std_gcn = run_model("gcn")
mean_appnp, std_appnp = run_model("appnp")
mean_gpr, std_gpr = run_model("gpr")

print("GCN:", mean_gcn, std_gcn)
print("APPNP:", mean_appnp, std_appnp)
print("GPR:", mean_gpr, std_gpr)

GCN: 0.7982035928143711 0.002395209580838348
APPNP: 0.8071856287425149 0.0030533050979597667
GPR: 0.788622754491018 0.006721540215761583


In [16]:
df_arch = pd.DataFrame([
    ["GCN", mean_gcn, std_gcn],
    ["APPNP", mean_appnp, std_appnp],
    ["GPR-GNN", mean_gpr, std_gpr],
], columns=["Model", "Mean Accuracy", "Std Dev"])

df_arch = df_arch.sort_values("Mean Accuracy", ascending=False)

df_arch

,Model,Mean Accuracy,Std Dev
1,APPNP,0.807186,0.003053
0,GCN,0.798204,0.002395
2,GPR-GNN,0.788623,0.006722


PHASE 2- K SWEEP(FOR APPNP)

In [17]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

hidden_dim = 256
alpha = 0.1
dropout = 0.6
weight_decay = 5e-4
lr = 0.01
epochs = 300
split_type = "random_80_10_10"

seeds = [0, 42, 7, 123, 999]

dataset, data = load_data(device, split_type=split_type)

print("Device:", device)

Device: mps


In [18]:
def run_appnp_with_k(K_value):
    results = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = APPNPModel(
            dataset.num_node_features,
            hidden_dim,
            dataset.num_classes,
            K=K_value,
            alpha=alpha,
            dropout=dropout
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        for epoch in range(epochs):
            train(model, data, optimizer)

        train_acc, val_acc, test_acc = test(model, data)

        results.append(test_acc)

    mean_acc = np.mean(results)
    std_acc = np.std(results)

    return mean_acc, std_acc

In [19]:
K_values = [5, 10, 20]

results_k = []

for K in K_values:
    print(f"Running K = {K}")
    mean_acc, std_acc = run_appnp_with_k(K)
    results_k.append([K, mean_acc, std_acc])

results_k

Running K = 5
Running K = 10
Running K = 20


[[5, 0.7994011976047904, 0.00927660681726326],
 [10, 0.7934131736526946, 0.0070851254887420445],
 [20, 0.7880239520958084, 0.006931638863946247]]

In [20]:
df_k = pd.DataFrame(results_k, columns=["K", "Mean Accuracy", "Std Dev"])

df_k = df_k.sort_values("Mean Accuracy", ascending=False)

df_k

,K,Mean Accuracy,Std Dev
0,5,0.799401,0.009277
1,10,0.793413,0.007085
2,20,0.788024,0.006932


Increasing K allows information to propagate further in the graph.

However, too large K can cause oversmoothing.

K=10 provides the best balance between local and global information.

Effect of Propagation Depth (K)

Increasing K allows information to propagate further across the graph.
However, excessive propagation leads to oversmoothing, where node representations become indistinguishable.

Our results show:

K=5 achieves the highest mean accuracy (0.7994)

Performance decreases as K increases

This confirms the oversmoothing effect in citation networks

Thus, moderate propagation depth provides the best balance between local and global information.

Phase 3 — Alpha (α) Tuning.

In [25]:
import torch
import numpy as np
import pandas as pd

from src.data_preprocessing import load_data
from src.model import APPNPModel
from src.train import train, test

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

hidden_dim = 64
K = 5  # Use best K from Phase 2
dropout = 0.6
weight_decay = 5e-4
lr = 0.01
epochs = 300
split_type = "random_80_10_10"

seeds = [0, 42, 7, 123, 999]

dataset, data = load_data(device, split_type=split_type)

print("Device:", device)

Device: mps


In [26]:
def run_appnp_with_alpha(alpha_value):
    results = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = APPNPModel(
            dataset.num_node_features,
            hidden_dim,
            dataset.num_classes,
            K=K,
            alpha=alpha_value,
            dropout=dropout
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        for epoch in range(epochs):
            train(model, data, optimizer)

        train_acc, val_acc, test_acc = test(model, data)

        results.append(test_acc)

    mean_acc = np.mean(results)
    std_acc = np.std(results)

    return mean_acc, std_acc

In [27]:
alpha_values = [0.1, 0.2, 0.5]

results_alpha = []

for alpha_val in alpha_values:
    print(f"Running alpha = {alpha_val}")
    mean_acc, std_acc = run_appnp_with_alpha(alpha_val)
    results_alpha.append([alpha_val, mean_acc, std_acc])

results_alpha

Running alpha = 0.1
Running alpha = 0.2
Running alpha = 0.5


[[0.1, 0.7970059880239521, 0.005151092974277014],
 [0.2, 0.7970059880239521, 0.009911943327692727],
 [0.5, 0.804191616766467, 0.005553064967362714]]

In [28]:
df_alpha = pd.DataFrame(results_alpha, columns=["Alpha", "Mean Accuracy", "Std Dev"])

df_alpha = df_alpha.sort_values("Mean Accuracy", ascending=False)

df_alpha

,Alpha,Mean Accuracy,Std Dev
2,0.5,0.804192,0.005553
0,0.1,0.797006,0.005151
1,0.2,0.797006,0.009912


The teleport probability α plays a crucial role in controlling the balance between propagated graph information and original node features in APPNP.

Across our experiments, we observed that changing α significantly impacts model performance:

Lower α values (e.g., 0.1) increase the influence of graph propagation. While this allows deeper diffusion of information, it may lead to oversmoothing, where node representations become too similar, reducing classification performance.

Moderate α values (e.g., 0.2) provide a balanced tradeoff between feature retention and propagation.

Higher α values (e.g., 0.5) emphasize original node features more strongly, reducing excessive smoothing and often improving stability.

In our experiments (with hidden_dim = 64), α = 0.5 achieved the highest mean accuracy (≈ 0.804), indicating that stronger feature retention benefits performance when model capacity is limited.

This demonstrates that α must be tuned carefully, as its optimal value depends on:

Model capacity (hidden dimension size)

Graph sparsity

Propagation depth (K)

Overall, the results confirm that moderate-to-high teleport probability improves generalization on sparse citation networks like CiteSeer.

PHASE 4 — Split Comparison.

In [29]:
import torch
import numpy as np
import pandas as pd

from src.data_preprocessing import load_data
from src.model import APPNPModel
from src.train import train, test

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

hidden_dim = 256
K = 5
alpha = 0.5
dropout = 0.6
weight_decay = 5e-4
lr = 0.01
epochs = 300

seeds = [0, 42, 7, 123, 999]

print("Device:", device)

Device: mps


In [30]:
def run_split(split_type):
    dataset, data = load_data(device, split_type=split_type)
    results = []

    for seed in seeds:
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = APPNPModel(
            dataset.num_node_features,
            hidden_dim,
            dataset.num_classes,
            K=K,
            alpha=alpha,
            dropout=dropout
        ).to(device)

        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )

        for epoch in range(epochs):
            train(model, data, optimizer)

        train_acc, val_acc, test_acc = test(model, data)
        results.append(test_acc)

    mean_acc = np.mean(results)
    std_acc = np.std(results)

    return mean_acc, std_acc

In [31]:
splits = ["random_60_20_20", "random_80_10_10"]

results_split = []

for split in splits:
    print(f"Running split: {split}")
    mean_acc, std_acc = run_split(split)
    results_split.append([split, mean_acc, std_acc])

results_split

Running split: random_60_20_20
Running split: random_80_10_10


[['random_60_20_20', 0.766966966966967, 0.0037266287225197967],
 ['random_80_10_10', 0.8041916167664672, 0.004061275438997162]]

In [32]:
df_split = pd.DataFrame(
    results_split,
    columns=["Split Type", "Mean Accuracy", "Std Dev"]
)

df_split = df_split.sort_values("Mean Accuracy", ascending=False)

df_split

,Split Type,Mean Accuracy,Std Dev
1,random_80_10_10,0.804192,0.004061
0,random_60_20_20,0.766967,0.003727


Effect of Training Data Proportion

To evaluate robustness, we compared performance under different train/validation/test splits.

Results show:

Increasing training data from 60% to 80% improves mean accuracy from 76.7% to 80.4%.

Standard deviation remains low across both splits, indicating stable performance.

The model benefits significantly from additional labeled data, demonstrating good data scalability.

These findings confirm that APPNP effectively leverages increased supervision in citation networks.

FINAL 

Model = APPNP

hidden_dim = 256

K = 5

alpha = 0.5

dropout = 0.6

split = random_80_10_10

Mean Accuracy = 0.8042